In [ ]:
# === ContentFactory YouTube VIDEO Worker ===
# Worker: iheuko119@gmail.com

import os
import sys
import json
from pathlib import Path

WORKER_EMAIL = "iheuko119@gmail.com"
print("[CF_BOOT] cell_started account=" + WORKER_EMAIL, flush=True)
print("[CF_BOOT] python_version python=" + sys.version.replace("\n", " "), flush=True)
print("[CF_BOOT] cwd cwd=" + str(Path.cwd()), flush=True)
print("[CF_BOOT] drive_mount_start", flush=True)
import os
import sys
import json
import time
import random
import shutil
import subprocess
import shlex
from datetime import datetime, timezone
from google.colab import drive

BOOT_STATUS_LOCAL_PATH = "/content/content_factory_boot_status_local.json"
MOUNT_JITTER_SECONDS = int(os.environ.get("CONTENT_FACTORY_DRIVE_MOUNT_JITTER_SECONDS", "120") or 120)
MOUNT_MAX_ATTEMPTS = int(os.environ.get("CONTENT_FACTORY_DRIVE_MOUNT_MAX_ATTEMPTS", "8") or 8)
MOUNT_BACKOFF_SECONDS = [15, 30, 60, 90, 120]


def _write_boot_status_local(*, attempt: int, error: str | None = None) -> None:
    payload = {
        "worker_email": WORKER_EMAIL,
        "stage": "drive_mount",
        "attempt": attempt,
        "error": error,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    try:
        with open(BOOT_STATUS_LOCAL_PATH, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2)
    except OSError as exc:
        print(f"[drive_mount] boot_status_local_write_failed error={exc!r}", flush=True)


def _is_mountpoint(path: str) -> bool:
    return subprocess.run(
        ["bash", "-lc", f"mountpoint -q {shlex.quote(path)}"],
        capture_output=True,
        text=True,
    ).returncode == 0


def _clean_stale_mount(mountpoint: str) -> None:
    print(f"[drive_mount] cleaning stale mount state at {mountpoint}")
    if _is_mountpoint(mountpoint):
        subprocess.run(
            ["bash", "-lc", f"fusermount -u {shlex.quote(mountpoint)} || umount -l {shlex.quote(mountpoint)} || true"],
            capture_output=True,
            text=True,
        )
    else:
        subprocess.run(
            ["bash", "-lc", f"fusermount -u {shlex.quote(mountpoint)} || umount -l {shlex.quote(mountpoint)} || true"],
            capture_output=True,
            text=True,
        )
    subprocess.run(["bash", "-lc", "pkill -f drivefs || true"], capture_output=True, text=True)
    if os.path.exists(mountpoint) and not _is_mountpoint(mountpoint):
        print(f"[drive_mount] removing stale folder {mountpoint}")
        shutil.rmtree(mountpoint, ignore_errors=True)
    os.makedirs(mountpoint, exist_ok=True)


def _attempt_drive_mount(mountpoint: str, *, force_remount: bool) -> None:
    if force_remount:
        drive.mount(mountpoint, force_remount=True)
    else:
        drive.mount(mountpoint)


def safe_mount_google_drive(mountpoint: str = "/content/drive") -> None:
    jitter = random.uniform(0, max(0, MOUNT_JITTER_SECONDS))
    print(f"[drive_mount] startup_jitter_seconds={jitter:.1f}", flush=True)
    if jitter > 0:
        time.sleep(jitter)

    if _is_mountpoint(mountpoint):
        print(f"[drive_mount] Google Drive already mounted at {mountpoint}", flush=True)
        _write_boot_status_local(attempt=0, error=None)
        return

    last_error = ""
    for attempt in range(1, MOUNT_MAX_ATTEMPTS + 1):
        try:
            if _is_mountpoint(mountpoint):
                print("[drive_mount] Google Drive mount OK (mountpoint detected)", flush=True)
                _write_boot_status_local(attempt=attempt, error=None)
                return

            stale = os.path.exists(mountpoint) and not _is_mountpoint(mountpoint)
            if stale or attempt > 1:
                _clean_stale_mount(mountpoint)
            else:
                os.makedirs(mountpoint, exist_ok=True)

            force_remount = bool(stale or attempt > 1)
            print(
                f"[drive_mount] attempt={attempt} mounting at {mountpoint} force_remount={force_remount}",
                flush=True,
            )
            _attempt_drive_mount(mountpoint, force_remount=force_remount)

            if _is_mountpoint(mountpoint):
                print("[drive_mount] Google Drive mount OK", flush=True)
                _write_boot_status_local(attempt=attempt, error=None)
                return

            raise RuntimeError(f"{mountpoint} is not a mountpoint after drive.mount")
        except Exception as exc:
            last_error = repr(exc)
            print(f"[drive_mount] attempt={attempt} failed error={last_error}", flush=True)
            _write_boot_status_local(attempt=attempt, error=last_error)
            if attempt >= MOUNT_MAX_ATTEMPTS:
                print("DRIVE_MOUNT_FAILED_RETRY_EXHAUSTED", flush=True)
                print(f"account={WORKER_EMAIL}", flush=True)
                print(f"attempts={MOUNT_MAX_ATTEMPTS}", flush=True)
                print(
                    "action=Restart runtime and run this worker again later. Do not relaunch all workers at once.",
                    flush=True,
                )
                raise RuntimeError(
                    f"DRIVE_MOUNT_FAILED_RETRY_EXHAUSTED account={WORKER_EMAIL} attempts={MOUNT_MAX_ATTEMPTS}. "
                    "Restart runtime and run this worker again later. Do not relaunch all workers at once."
                ) from exc
            backoff_idx = min(attempt - 1, len(MOUNT_BACKOFF_SECONDS) - 1)
            retry_in = float(MOUNT_BACKOFF_SECONDS[backoff_idx]) + random.uniform(0, 15)
            print(f"[drive_mount] retry_in_seconds={retry_in:.1f}", flush=True)
            time.sleep(retry_in)

    raise RuntimeError(
        f"DRIVE_MOUNT_FAILED_RETRY_EXHAUSTED account={WORKER_EMAIL} attempts={MOUNT_MAX_ATTEMPTS} last_error={last_error}"
    )


safe_mount_google_drive("/content/drive")
print("[CF_BOOT] drive_mount_ok", flush=True)

!apt-get update -qq
!apt-get install -y -qq ffmpeg

import subprocess

ROOT = Path("/content/drive/MyDrive/ContentFactory_YouTube")
BOOTSTRAP_PATH = ROOT / "scripts" / "youtube_video_bootstrap_colab.py"
SCRIPT_PATH = ROOT / "scripts" / "youtube_video_worker_colab.py"
BOOT_STATUS_PATH = ROOT / "logs" / "colab_boot_status.json"

def write_cell_boot_status(stage, **updates):
    ROOT.joinpath("logs").mkdir(parents=True, exist_ok=True)
    payload = {
        "account": WORKER_EMAIL,
        "started_at": updates.pop("started_at", None),
        "last_stage": stage,
        "last_stage_at": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
        "ok": False,
        "error_stage": None,
        "error": None,
        "traceback": None,
        "heartbeat_path": "",
        "heartbeat_written_once": False,
        "worker_main_loop_started": False,
    }
    if BOOT_STATUS_PATH.exists():
        try:
            payload.update(json.loads(BOOT_STATUS_PATH.read_text(encoding="utf-8")))
        except Exception:
            pass
    payload.update(updates)
    payload["account"] = WORKER_EMAIL
    payload["last_stage"] = stage
    payload["last_stage_at"] = __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat()
    BOOT_STATUS_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return BOOT_STATUS_PATH

print("WORKER_EMAIL:", WORKER_EMAIL)
print("ROOT exists:", ROOT.exists(), ROOT)
print("BOOTSTRAP exists:", BOOTSTRAP_PATH.exists(), BOOTSTRAP_PATH)
print("SCRIPT exists:", SCRIPT_PATH.exists(), SCRIPT_PATH)
print("[CF_BOOT] project_root_detected root=" + str(ROOT) + " root_exists=" + str(ROOT.exists()), flush=True)
try:
    write_cell_boot_status("project_root_detected")
except Exception as exc:
    print("[CF_BOOT_ERROR] stage=cell_boot_status", flush=True)
    print("[CF_BOOT_ERROR] exception=" + repr(exc), flush=True)

if not ROOT.exists():
    raise RuntimeError(
        "ContentFactory_YouTube не найден. "
        "Проверь, что для этого Google-аккаунта создан shortcut в My Drive."
    )

if not BOOTSTRAP_PATH.exists():
    raise RuntimeError(
        "youtube_video_bootstrap_colab.py не найден. "
        "Сначала запусти setup-colab-workers на Windows."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "youtube_video_worker_colab.py не найден. "
        "Сначала запусти setup-colab-workers на Windows."
    )

os.environ["CONTENT_FACTORY_WORKER_EMAIL"] = WORKER_EMAIL
os.environ["CONTENT_FACTORY_YOUTUBE_ROOT"] = str(ROOT)
os.environ["CONTENT_FACTORY_VIDEO_QUEUE_MODE"] = "1"
os.environ["CONTENT_FACTORY_MAX_JOBS_PER_RUN"] = "0"
os.environ["CONTENT_FACTORY_POLL_SECONDS"] = "10"
os.environ["CONTENT_FACTORY_IDLE_TIMEOUT_MIN"] = "15"
os.environ["CONTENT_FACTORY_IDLE_EXIT_SECONDS"] = "900"
os.environ["CONTENT_FACTORY_SELF_RECLAIM_STALE_MINUTES"] = "10"
os.environ["CONTENT_FACTORY_SELF_RECLAIM_MAX_ATTEMPTS"] = "3"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CONTENT_FACTORY_REQUIRE_T4"] = "0"
print("[CF_BOOT] env_loaded account=" + WORKER_EMAIL, flush=True)
try:
    write_cell_boot_status("env_loaded")
except Exception as exc:
    print("[CF_BOOT_ERROR] stage=cell_boot_status", flush=True)
    print("[CF_BOOT_ERROR] exception=" + repr(exc), flush=True)

import shutil

gpu_name = ""
nvidia_smi = shutil.which("nvidia-smi")

if not nvidia_smi:
    print("GPU: not available")
    message = "GPU not available. In Colab use Runtime -> Change runtime type -> GPU."
    if os.environ.get("CONTENT_FACTORY_REQUIRE_T4") == "1":
        raise RuntimeError(message)
    print("[WARN]", message)
else:
    gpu = subprocess.run(
        [nvidia_smi, "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    gpu_name = gpu.stdout.strip().splitlines()[0].strip() if gpu.returncode == 0 and gpu.stdout.strip() else ""
    print("GPU:", gpu_name or "not available")

    if not gpu_name:
        message = "GPU not available. In Colab use Runtime -> Change runtime type -> GPU."
        if os.environ.get("CONTENT_FACTORY_REQUIRE_T4") == "1":
            raise RuntimeError(message)
        print("[WARN]", message)
    elif "T4" not in gpu_name.upper():
        message = f"GPU is not T4: {gpu_name}. Continuing because CONTENT_FACTORY_REQUIRE_T4=0."
        if os.environ.get("CONTENT_FACTORY_REQUIRE_T4") == "1":
            raise RuntimeError(message)
        print("[WARN]", message)

print("[CF_BOOT] worker_import_start worker_script=" + str(SCRIPT_PATH), flush=True)
%run "/content/drive/MyDrive/ContentFactory_YouTube/scripts/youtube_video_bootstrap_colab.py" --story-slug "Becoming_A_Slut_Wife_Alma" --worker-email "iheuko119@gmail.com" --max-jobs-per-run "0" --idle-timeout-min "15" --poll-seconds "10"
